# 대피도 YOLO Round 2 — Recall 중심 개선 학습

현재 완료 모델의 약점인 **Recall**, 특히 `exit / stair / you_are_here`를 개선하기 위한 다음 학습 노트북입니다.

목표(프로젝트 내부 기준):
- mAP50 ≥ 0.80, mAP50-95 ≥ 0.55
- Precision ≥ 0.85, Recall ≥ 0.85
- exit / stair / you_are_here Recall ≥ 0.90

> 목표 달성은 보장되지 않습니다. 가장 중요한 입력은 **서로 다른 실제 원본 + 사람이 검수한 라벨**입니다.

In [ ]:
# 0. Google Drive 연결 및 작업공간 준비
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, zipfile, json, os, sys

WORK = Path('/content/evac_round2')
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
print(WORK)

## 1. 이 키트 ZIP 업로드
`evac_next_training_kit.zip`을 업로드합니다. 현재 `best.pt`가 키트 안에 들어 있으므로 기존 80 epoch를 다시 돌리지 않습니다.

In [ ]:
from google.colab import files
uploaded = files.upload()
kit_zip = next(Path(k) for k in uploaded if k.endswith('.zip'))
with zipfile.ZipFile(kit_zip) as z:
    z.extractall(WORK)
KIT = WORK / 'evac_next_training_kit'
assert (KIT/'seed_model'/'best.pt').exists()
print('KIT:', KIT)

## 2. 기존 `ml` 프로젝트 복원
권장: Drive의 최신 `evac_ml_checkpoint_*.zip`을 복원합니다. 없으면 `ml_complete_200plus.zip`을 업로드해도 됩니다.

In [ ]:
checkpoint_dir = Path('/content/drive/MyDrive/evacuation_checkpoints')
checkpoints = sorted(checkpoint_dir.glob('evac_ml_checkpoint_*.zip'), key=lambda p:p.stat().st_mtime) if checkpoint_dir.exists() else []
if checkpoints:
    src = checkpoints[-1]
    print('checkpoint 사용:', src)
else:
    print('체크포인트가 없습니다. ml_complete_200plus.zip을 업로드하세요.')
    up = files.upload()
    src = next(Path(k) for k in up if 'ml_complete' in k and k.endswith('.zip'))

PROJ = WORK/'project'
PROJ.mkdir(exist_ok=True)
with zipfile.ZipFile(src) as z: z.extractall(PROJ)
ML = PROJ/'ml'
if not ML.exists():
    # 일부 ZIP은 wrapper 없이 내용만 들어있을 수 있음
    candidates=list(PROJ.rglob('import_real.py'))
    assert candidates, 'ml 프로젝트를 찾지 못했습니다.'
    ML=candidates[0].parent
DATASET=ML/'dataset'
print('ML=',ML,'DATASET=',DATASET)

In [ ]:
# 3. 패키지 설치 및 변수 준비
os.chdir(ML)
!pip install -q -r requirements.txt
from ultralytics import YOLO
import torch
NAMES=['exit','stair','elevator','extinguisher','hydrant','you_are_here','door','room']
SEED_MODEL = KIT/'seed_model'/'best.pt'
IMAGE_SIZE=960
BATCH = 8 if torch.cuda.is_available() else 2
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('seed:', SEED_MODEL)

## 4. 새 실제 원본 수집
가장 중요한 단계입니다. 같은 13개 원본의 증강만 더 만들지 말고 **서로 다른 실제 대피도 원본**을 추가하세요.

권장 목표: 새 원본 그룹 50개 이상. `stair`가 포함된 대피도를 우선 수집하세요.

프로젝트의 Wikimedia 수집기를 사용할 수 있습니다. 네트워크/라이선스 정책에 따라 일부 파일은 건너뜁니다.

In [ ]:
NEW_RAW = WORK/'new_real_raw'
# 필요할 때만 실행하세요. 이미 새 원본 이미지가 있다면 이 셀을 건너뛰고 아래 업로드 단계로 가면 됩니다.
# !python crawl_wikimedia_evacuation.py --out {NEW_RAW} --target 200 --depth 6 --clean
print('새 원본을 준비할 위치:', NEW_RAW)

## 5. 새 이미지 자동 밑라벨 생성 → 다운로드 → 사람 검수
자동 밑라벨은 정답이 아닙니다. **누락 추가 / 오탐 삭제 / 클래스 수정**을 모두 끝낸 뒤 재업로드하세요.

특히 `stair`, `exit`, `you_are_here`를 집중 검수합니다.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil

NEW_RAW = Path("/content/evac_round2/new_real_raw")
shutil.rmtree(NEW_RAW, ignore_errors=True)
NEW_RAW.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()

zip_name = next(name for name in uploaded if name.lower().endswith(".zip"))

with zipfile.ZipFile(zip_name) as z:
    z.extractall(NEW_RAW)

print("압축 해제 완료:", NEW_RAW)
print("이미지 폴더:", NEW_RAW / "images")
print("이미지 수:", len(list((NEW_RAW / "images").glob("*"))))

In [ ]:
from pathlib import Path

KIT = Path("/content/evac_round2/evac_next_training_kit")

symbols_py = KIT / "scripts" / "symbols.py"

symbols_py.write_text(
'''NAMES = [
    "exit",
    "stair",
    "elevator",
    "extinguisher",
    "hydrant",
    "you_are_here",
    "door",
    "room",
]
''',
    encoding="utf-8"
)

print("생성 완료:", symbols_py)
print(symbols_py.read_text())

In [ ]:
# 새 이미지 폴더를 직접 업로드한 ZIP으로 쓸 경우 여기서 압축 해제하세요.
# 예: images/ 폴더 또는 이미지 파일들이 들어 있는 ZIP
# up = files.upload(); ...

# NEW_IMAGES를 실제 이미지 폴더로 맞추세요.
NEW_IMAGES = NEW_RAW/'images'
PRE = WORK/'round2_prelabeled'
if NEW_IMAGES.exists() and any(NEW_IMAGES.glob('*')):
    shutil.rmtree(PRE,ignore_errors=True)
    !python {KIT/'scripts'/'label_assist_safe.py'} --model {SEED_MODEL} --images {NEW_IMAGES} --out {PRE} --conf 0.12 --imgsz {IMAGE_SIZE} --overwrite
    shutil.make_archive('/content/round2_prelabeled','zip',PRE)
    files.download('/content/round2_prelabeled.zip')
else:
    print('NEW_IMAGES에 이미지가 아직 없습니다. 새 원본 준비 후 실행하세요:', NEW_IMAGES)

## 6. 사람이 검수한 Round 2 ZIP 업로드 및 병합
검수 완료 ZIP은 `images/`, `labels/`, 가능하면 `classes.txt`, `split_manifest.json`을 포함해야 합니다.

`split_manifest.json`에서 train/val/test를 지정할 수 있습니다. 없으면 원본 그룹 단위 70/15/15로 자동 분할합니다.

In [ ]:
up=files.upload()
review_zip=next(Path(k) for k in up if k.endswith('.zip'))
REVIEWED=WORK/'reviewed_round2'
shutil.rmtree(REVIEWED,ignore_errors=True); REVIEWED.mkdir()
with zipfile.ZipFile(review_zip) as z: z.extractall(REVIEWED)
!python {KIT/'scripts'/'import_round2.py'} --reviewed {REVIEWED} --dataset {DATASET} --prefix real2_

## 7. 기존 데이터 + 새 데이터 목록 구성
기존 208장의 라벨을 **사람이 모두 검수했다면** `OLD_REAL_REVIEWED=True`로 바꾸세요. 자동 밑라벨 그대로라면 반드시 False로 유지합니다.

새 val/test는 이번 Round 2의 새로운 원본 그룹만 사용합니다.

In [ ]:
OLD_REAL_REVIEWED = False  # 사람이 기존 208장을 전부 검수했다면 True
cmd=f"python {KIT/'scripts'/'combine_lists.py'} --dataset {DATASET}"
if OLD_REAL_REVIEWED: cmd += ' --old-real-reviewed'
!{cmd}

## 8. 핵심 클래스 가중 train list 생성
실제 데이터와 `exit/stair/you_are_here`가 포함된 이미지를 더 자주 보게 합니다. `stair` 이미지는 추가 가중합니다.

이는 새 실제 다양성을 대신하지 않습니다. 한 원본을 과도하게 반복하지 않도록 반복 상한을 6으로 둡니다.

In [ ]:
!python {KIT/'scripts'/'audit_and_balance.py'}   --dataset {DATASET}   --train-list {DATASET/'round2_train_base.txt'}   --out {DATASET/'round2_train_weighted.txt'}   --real-base-repeat 2 --critical-extra 2 --stair-extra 2 --max-repeat 6

In [ ]:
# 9. Round 2 YAML 생성
ROUND2_YAML=ML/'data_round2.yaml'
!python {KIT/'scripts'/'make_round2_yaml.py'} --dataset {DATASET} --out {ROUND2_YAML}

In [ ]:
from pathlib import Path
import random
import subprocess
import sys

# 기존 변수 사용
DATASET = Path("/content/evac_round2/project/ml/dataset")
KIT = Path("/content/evac_round2/evac_next_training_kit")
ML = Path("/content/evac_round2/project/ml")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

print("=== Round 2 split 복구 시작 ===")

# --------------------------------------------------
# 1. 실제로 존재하는 새 Round2 이미지 찾기
# --------------------------------------------------
new_images = []

for split_dir in ["train", "val", "test"]:
    d = DATASET / "images" / split_dir
    if d.exists():
        for p in d.iterdir():
            if (
                p.is_file()
                and p.suffix.lower() in IMG_EXTS
                and p.name.startswith("real2_")
            ):
                new_images.append(p.resolve())

new_images = sorted(set(new_images))

print("찾은 real2_ 이미지:", len(new_images))

if len(new_images) < 3:
    raise RuntimeError(
        f"Round2 실제 이미지가 {len(new_images)}장뿐입니다. "
        "6번 병합이 정상적으로 완료되었는지 확인해야 합니다."
    )

# --------------------------------------------------
# 2. 이미지마다 대응하는 label 존재 확인
# --------------------------------------------------
valid_images = []
missing_labels = []

for im in new_images:
    # .../images/train/a.jpg -> .../labels/train/a.txt
    parts = list(im.parts)
    idx = parts.index("images")
    parts[idx] = "labels"

    lab = Path(*parts).with_suffix(".txt")

    if lab.exists():
        valid_images.append(im)
    else:
        missing_labels.append((im, lab))

print("정상 이미지+라벨:", len(valid_images))
print("라벨 누락:", len(missing_labels))

if missing_labels:
    for im, lab in missing_labels[:10]:
        print("누락:", im)
        print(" ->", lab)

if len(valid_images) < 3:
    raise RuntimeError("학습 가능한 Round2 이미지/라벨 쌍이 부족합니다.")

# --------------------------------------------------
# 3. 현재 11장 규모에 맞춰 70/15/15 재분할
#
# 파일을 실제로 이동하지 않고 txt 목록만 나눕니다.
# YOLO는 절대경로 txt를 읽을 수 있으므로 안전합니다.
# --------------------------------------------------
rng = random.Random(20260815)
valid_images = valid_images.copy()
rng.shuffle(valid_images)

n = len(valid_images)

n_test = max(1, round(n * 0.15))
n_val  = max(1, round(n * 0.15))

# train은 반드시 1장 이상
if n_test + n_val >= n:
    n_test = 1
    n_val = 1

test_new = valid_images[:n_test]
val_new = valid_images[n_test:n_test + n_val]
train_new = valid_images[n_test + n_val:]

print()
print("새 Round2 분할")
print("train:", len(train_new))
print("val  :", len(val_new))
print("test :", len(test_new))

# --------------------------------------------------
# 4. 새 데이터 목록 다시 저장
# --------------------------------------------------
def write_list(path, items):
    path.write_text(
        "\n".join(str(x) for x in items) + "\n",
        encoding="utf-8",
    )

write_list(DATASET / "round2_train_new.txt", train_new)
write_list(DATASET / "round2_val_new.txt", val_new)
write_list(DATASET / "round2_test_new.txt", test_new)

# --------------------------------------------------
# 5. 기존 synthetic train + 새로운 Round2 train 결합
#
# 기존 208장이 완전 수동검수된 것이 아니라면
# real_ 로 시작하는 옛 자동라벨은 제외
# --------------------------------------------------
base_train = []

train_dir = DATASET / "images" / "train"

if train_dir.exists():
    for p in train_dir.iterdir():
        if not p.is_file():
            continue

        if p.suffix.lower() not in IMG_EXTS:
            continue

        # 이번 Round2는 별도로 추가
        if p.name.startswith("real2_"):
            continue

        # 기존 pseudo real 라벨은 제외
        if p.name.startswith("real_"):
            continue

        base_train.append(p.resolve())

base_train = sorted(set(base_train))
round2_train_base = base_train + train_new

write_list(
    DATASET / "round2_train_base.txt",
    round2_train_base
)

# 최종 val/test
write_list(
    DATASET / "round2_val.txt",
    val_new
)

write_list(
    DATASET / "round2_test.txt",
    test_new
)

print()
print("기존 학습 이미지:", len(base_train))
print("Round2 새 train:", len(train_new))
print("전체 train base:", len(round2_train_base))

# --------------------------------------------------
# 6. 핵심 클래스 weighted train 다시 생성
# --------------------------------------------------
cmd = [
    sys.executable,
    str(KIT / "scripts" / "audit_and_balance.py"),
    "--dataset", str(DATASET),
    "--train-list", str(DATASET / "round2_train_base.txt"),
    "--out", str(DATASET / "round2_train_weighted.txt"),
    "--real-base-repeat", "2",
    "--critical-extra", "2",
    "--stair-extra", "2",
    "--max-repeat", "6",
]

subprocess.run(cmd, check=True)

# --------------------------------------------------
# 7. YAML도 다시 생성
# --------------------------------------------------
ROUND2_YAML = ML / "data_round2.yaml"

cmd = [
    sys.executable,
    str(KIT / "scripts" / "make_round2_yaml.py"),
    "--dataset", str(DATASET),
    "--out", str(ROUND2_YAML),
]

subprocess.run(cmd, check=True)

# --------------------------------------------------
# 8. 실제 경로 최종 검증
# --------------------------------------------------
print("\n=== 최종 검증 ===")

for name in [
    "round2_train_weighted.txt",
    "round2_val.txt",
    "round2_test.txt",
]:
    p = DATASET / name

    lines = [
        x.strip()
        for x in p.read_text(encoding="utf-8").splitlines()
        if x.strip()
    ]

    existing = sum(Path(x).exists() for x in lines)

    print(
        name,
        "목록:", len(lines),
        "/ 실제 존재:", existing
    )

    assert len(lines) > 0, f"{name}가 비어 있습니다."
    assert existing == len(lines), f"{name}에 잘못된 경로가 있습니다."

print("\nYAML:")
print(ROUND2_YAML.read_text())

print("\n✅ Round2 train / val / test 경로 복구 완료")
print("이제 바로 10번 셀을 실행하세요.")

## 10. Stage A — 960px Recall 중심 미세조정
현재 `best.pt`에서 시작합니다. 이전 합성 1차 학습과 80 epoch 학습을 재실행하지 않습니다.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from google.colab import drive

# --------------------------------------------------
# Google Drive 연결
# --------------------------------------------------
drive.mount("/content/drive", force_remount=False)

# --------------------------------------------------
# Drive 저장 위치
# --------------------------------------------------
DRIVE_RUNS = Path(
    "/content/drive/MyDrive/evacuation_checkpoints/round2"
)
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)

RUN_NAME = "evac_round2_stage_a"

RUN_DIR = DRIVE_RUNS / RUN_NAME
LAST_DRIVE = RUN_DIR / "weights" / "last.pt"

print("========================================")
print("Stage A 저장 위치:", RUN_DIR)
print("최대 epoch: 50")
print("patience: 12")
print("========================================")

# --------------------------------------------------
# 기존 last.pt가 있으면 자동 이어학습
# --------------------------------------------------
if LAST_DRIVE.exists():

    print()
    print("✅ 기존 last.pt 발견")
    print("이어학습:", LAST_DRIVE)

    model = YOLO(str(LAST_DRIVE))

    stage_a = model.train(
        resume=True
    )

# --------------------------------------------------
# 없으면 현재 best.pt에서 Stage A 시작
# --------------------------------------------------
else:

    print()
    print("새 Stage A 학습 시작")
    print("시작 모델:", SEED_MODEL)

    model = YOLO(str(SEED_MODEL))

    stage_a = model.train(
        data=str(ROUND2_YAML),

        # ------------------------------------------
        # 시간 절약 버전
        # ------------------------------------------
        epochs=50,
        patience=12,

        imgsz=IMAGE_SIZE,
        batch=BATCH,

        optimizer="AdamW",
        lr0=8e-4,
        lrf=0.05,
        weight_decay=5e-4,

        warmup_epochs=2.0,
        cos_lr=True,

        # ------------------------------------------
        # augmentation
        # ------------------------------------------
        degrees=4.0,
        translate=0.06,
        scale=0.20,
        perspective=0.0015,

        hsv_h=0.01,
        hsv_s=0.18,
        hsv_v=0.20,

        fliplr=0.0,
        flipud=0.0,

        mosaic=0.15,
        mixup=0.0,
        close_mosaic=10,

        amp=True,
        cache=False,
        workers=2,

        # ------------------------------------------
        # Drive에 직접 저장
        # ------------------------------------------
        project=str(DRIVE_RUNS),
        name=RUN_NAME,
        exist_ok=True,

        seed=20260815,
    )

# --------------------------------------------------
# 학습 결과 위치
# --------------------------------------------------
BEST_A = RUN_DIR / "weights" / "best.pt"
LAST_A = RUN_DIR / "weights" / "last.pt"

print()
print("========================================")
print("Stage A 완료/중단 상태")
print("RUN :", RUN_DIR)
print("BEST:", BEST_A, BEST_A.exists())
print("LAST:", LAST_A, LAST_A.exists())
print("========================================")

## 11. Stage B — 낮은 학습률로 정밀 마무리
Stage A의 best를 이어받아 mosaic 없이 40 epoch 이내로 마무리합니다.

In [ ]:
model_b=YOLO(str(BEST_A))
stage_b=model_b.train(
    data=str(ROUND2_YAML), epochs=40, patience=15,
    imgsz=IMAGE_SIZE, batch=BATCH,
    optimizer='AdamW', lr0=2.5e-4, lrf=0.10, weight_decay=5e-4,
    warmup_epochs=1.0, cos_lr=True,
    degrees=2.0, translate=0.03, scale=0.10, perspective=0.0005,
    hsv_h=0.005, hsv_s=0.10, hsv_v=0.12,
    fliplr=0.0, flipud=0.0, mosaic=0.0, mixup=0.0,
    amp=True, cache=False, workers=2,
    project=str(ML/'runs'), name='evac_round2_stage_b', seed=20260815,
)
BEST_B=Path(stage_b.save_dir)/'weights'/'best.pt'
print(BEST_B)

## 12. 새 holdout test에서 목표 자동 판정
**최종 성공 판단은 test로 합니다.** test 데이터는 학습/모델 선택에 사용하지 마세요.

In [ ]:
# Stage A/B를 새 test에서 각각 확인하고 더 나은 후보를 선택합니다.
# 실제 프로젝트에서는 test를 반복 튜닝에 쓰지 말고, 최종 확인용으로만 사용하세요.
for tag,mp in [('A',BEST_A),('B',BEST_B)]:
    out=WORK/f'target_eval_{tag}.json'
    !python {KIT/'scripts'/'evaluate_targets.py'} --model {mp} --data {ROUND2_YAML} --imgsz {IMAGE_SIZE} --out {out}

### 중요
Ultralytics `val()`은 YAML의 `val:`을 기본 사용합니다. 완전히 독립된 `test:`를 평가하려면 아래 셀에서 `split='test'`를 명시합니다.

In [ ]:
from ultralytics import YOLO

def eval_test(model_path):
    m=YOLO(str(model_path))
    return m.val(data=str(ROUND2_YAML), split='test', imgsz=IMAGE_SIZE, verbose=False, plots=True)

ra=eval_test(BEST_A); rb=eval_test(BEST_B)
print('A test:', float(ra.box.mp), float(ra.box.mr), float(ra.box.map50), float(ra.box.map))
print('B test:', float(rb.box.mp), float(rb.box.mr), float(rb.box.map50), float(rb.box.map))
# mAP50 우선, 동률이면 Recall을 더 크게 반영
scoreA=float(ra.box.map50)+0.5*float(ra.box.mr)
scoreB=float(rb.box.map50)+0.5*float(rb.box.mr)
BEST_ROUND2=BEST_B if scoreB>=scoreA else BEST_A
print('선택 모델:',BEST_ROUND2)

## 13. 클래스별 목표 판정
`exit/stair/you_are_here` Recall 0.90을 포함해 판정합니다.

In [ ]:
r=eval_test(BEST_ROUND2)
idx=[int(x) for x in r.box.ap_class_index]
per={NAMES[c]:{'P':float(r.box.p[j]),'R':float(r.box.r[j]),'mAP50':float(r.box.ap50[j]),'mAP50-95':float(r.box.ap[j])} for j,c in enumerate(idx)}
overall={'P':float(r.box.mp),'R':float(r.box.mr),'mAP50':float(r.box.map50),'mAP50-95':float(r.box.map)}
checks={
 'P>=0.85':overall['P']>=.85,'R>=0.85':overall['R']>=.85,'mAP50>=0.80':overall['mAP50']>=.80,'mAP50-95>=0.55':overall['mAP50-95']>=.55,
 'exit R>=0.90':per.get('exit',{}).get('R',0)>=.90,'stair R>=0.90':per.get('stair',{}).get('R',0)>=.90,'you_are_here R>=0.90':per.get('you_are_here',{}).get('R',0)>=.90}
print(json.dumps({'overall':overall,'per_class':per,'checks':checks,'PASS':all(checks.values())},ensure_ascii=False,indent=2))

## 14. 최종 내보내기
목표를 통과하지 못하면 무조건 epoch만 더 늘리지 마세요. `round2_balance_report.json`과 클래스별 Recall을 보고 부족한 실제 원본/라벨을 추가한 뒤 Round 3로 넘어가는 것이 우선입니다.

In [ ]:
FINAL=YOLO(str(BEST_ROUND2))
onnx=FINAL.export(format='onnx',imgsz=IMAGE_SIZE,opset=12,simplify=True)
OUT=Path('/content/evac_model_round2_final');shutil.rmtree(OUT,ignore_errors=True);OUT.mkdir()
for p in [BEST_ROUND2,Path(onnx),ROUND2_YAML,DATASET/'round2_import_report.json',DATASET/'round2_balance_report.json']:
    if p.exists(): shutil.copy2(p,OUT/p.name)
(OUT/'target_result.json').write_text(json.dumps({'overall':overall,'per_class':per,'checks':checks,'PASS':all(checks.values())},ensure_ascii=False,indent=2),encoding='utf-8')
shutil.make_archive('/content/evac_model_round2_final','zip',OUT)
# Drive에도 보존
save=Path('/content/drive/MyDrive/evacuation_checkpoints');save.mkdir(parents=True,exist_ok=True)
shutil.copy2('/content/evac_model_round2_final.zip',save/'evac_model_round2_final.zip')
files.download('/content/evac_model_round2_final.zip')